In [0]:
sql_server   = dbutils.secrets.get(scope="retailpulse-scope", key="sql-server-vault")
sql_database = dbutils.secrets.get(scope="retailpulse-scope", key="db-name")
sql_username = dbutils.secrets.get(scope="retailpulse-scope", key="retailpulse-sql-username")
sql_password = dbutils.secrets.get(scope="retailpulse-scope", key="retailpulse-sql-password")


STORAGE_ACCOUNT_KEY = dbutils.secrets.get(scope ="retailpulse-scope", key= "retailpulse-account-key")
STORAGE_ACCOUNT_NAME = "retailpulsestorage" 
CONTAINER_NAME       = "raw-data"


spark.conf.set(
    f"fs.azure.account.key.{STORAGE_ACCOUNT_NAME}.blob.core.windows.net",
    STORAGE_ACCOUNT_KEY
)

In [0]:
mount_path = dbutils.fs.ls(f"wasbs://{CONTAINER_NAME}@{STORAGE_ACCOUNT_NAME}.blob.core.windows.net")
path = mount_path[1]

In [0]:
jdbc_url = (
    f"jdbc:sqlserver://{sql_server}:1433;"
    f"database={sql_database};"
    f"encrypt=true;"
    f"trustServerCertificate=false;"
    f"hostNameInCertificate=*.database.windows.net;"
    f"loginTimeout=30"
)

connection_properties = {
    "user":     sql_username,
    "password": sql_password,
    "driver":   "com.microsoft.sqlserver.jdbc.SQLServerDriver"
}

# Load Silver layer
df_clean = spark.read.parquet(f"{path.path}/clean_transactions")

print(f"Silver rows loaded: {df_clean.count():,}")

In [0]:
for table in ["dim_customers", "dim_products", "dim_date"]:
    df = spark.read.jdbc(
        url=jdbc_url,
        table=table,
        properties=connection_properties
    )
    print(f"\n{table} columns:")
    print(df.columns)

In [0]:
from pyspark.sql.functions import (
    col, year, month, dayofmonth
)

df_fact = df_clean \
    .withColumn("date_key",
        (year("InvoiceDate") * 10000 +
         month("InvoiceDate") * 100 +
         dayofmonth("InvoiceDate")).cast("int")
    ) \
    .select(
        col("Invoice").alias("invoice"),
        col("Customer ID").alias("customer_id"),
        col("StockCode").alias("stock_code"),
        col("date_key"),
        col("InvoiceDate").alias("invoice_date"),
        col("Quantity").alias("quantity"),
        col("Price").alias("unit_price"),
        col("total_amount"),
        col("Country").alias("country")
    )

print(f"Fact table rows: {df_fact.count():,}")
print(f"\nNull check:")
print(f"  Null customer_id: {df_fact.filter(col('customer_id').isNull()).count():,}")
print(f"  Null stock_code:  {df_fact.filter(col('stock_code').isNull()).count():,}")
print(f"  Null date_key:    {df_fact.filter(col('date_key').isNull()).count():,}")

display(df_fact.limit(5))

In [0]:
from pyspark.sql.functions import monotonically_increasing_id

def execute_ddl(statement):
    conn = (spark._jvm.java.sql.DriverManager.getConnection(
        jdbc_url, sql_username, sql_password
    ))
    stmt = conn.createStatement()
    stmt.execute(statement)
    stmt.close()
    conn.close()

execute_ddl("TRUNCATE TABLE fact_transactions")
print(" Truncated")

df_indexed = df_fact.withColumn("row_id", monotonically_increasing_id())

CHUNK_SIZE  = 100_000
total_rows  = df_fact.count()
num_chunks  = (total_rows // CHUNK_SIZE) + 1

print(f"Writing {total_rows:,} rows in {num_chunks} chunks...")

for i in range(num_chunks):
    low  = i * CHUNK_SIZE
    high = low + CHUNK_SIZE
    
    chunk = df_indexed.filter(
        (col("row_id") >= low) & (col("row_id") < high)
    ).drop("row_id")
    
    chunk.write \
        .format("jdbc") \
        .option("url",       jdbc_url) \
        .option("dbtable",   "fact_transactions") \
        .option("user",      sql_username) \
        .option("password",  sql_password) \
        .option("driver",    "com.microsoft.sqlserver.jdbc.SQLServerDriver") \
        .option("batchsize", "5000") \
        .mode("append") \
        .save()
    
    print(f" Chunk {i+1}/{num_chunks} — rows {low:,} to {high:,}")

print("\nAll chunks written successfully")

In [0]:
df_verify = spark.read \
    .format("jdbc") \
    .option("url",   jdbc_url) \
    .option("query", """
        SELECT 
            COUNT(*)                    AS total_rows,
            COUNT(DISTINCT invoice)     AS unique_invoices,
            COUNT(DISTINCT customer_id) AS unique_customers,
            ROUND(SUM(total_amount), 2) AS total_revenue,
            MIN(invoice_date)           AS earliest_date,
            MAX(invoice_date)           AS latest_date
        FROM fact_transactions
    """) \
    .option("user",     sql_username) \
    .option("password", sql_password) \
    .option("driver",   "com.microsoft.sqlserver.jdbc.SQLServerDriver") \
    .load()

display(df_verify)

In [0]:
print(df_fact.count())
print(sql_username)